# MovieLens Analytics with Spark SQL and Hive

This notebook reads MovieLens CSV files from the student's HDFS directory, queries temporary views with Spark SQL, and saves a curated result as a persistent managed table in the shared Hive catalog.

## Learning objectives

- Register DataFrames as session-scoped temporary views.
- Express aggregations and joins in Spark SQL.
- Distinguish a temporary view from a persistent Hive table.
- Verify that another Hive-compatible session can discover the saved table.

## 1. Start the required services

This notebook requires HDFS, the Spark standalone cluster, and the external Hive Metastore described in the course setup notes. Start them in a WSL terminal:

```bash
start-dfs.sh
/opt/spark/sbin/start-master.sh
/opt/spark/sbin/start-worker.sh "spark://$(hostname):7077"

nohup hive --service metastore \
  > "$HOME/hive-logs/metastore.log" 2>&1 &

jps
ss -lnt | grep ':9083'
hdfs dfsadmin -report
```

Port 9083 must be listening before Spark starts. The external metastore lets separate Spark sessions and Hive clients share catalog metadata. HiveServer2 is not required for Spark to contact the metastore directly.

## 2. Prepare the MovieLens data in HDFS

The Windows files are expected at `C:\data\movies.csv` and `C:\data\ratings.csv`. WSL exposes that directory as `/mnt/c/data`.

Run the following in a **WSL terminal**. It creates a `movielens` directory inside your HDFS user directory, then creates separate `movies` and `ratings` input directories.

```bash
test -f /mnt/c/data/movies.csv
test -f /mnt/c/data/ratings.csv

hdfs dfs -mkdir -p \
  "/user/$USER/movielens/movies" \
  "/user/$USER/movielens/ratings"

hdfs dfs -put -f \
  /mnt/c/data/movies.csv \
  "/user/$USER/movielens/movies/"

hdfs dfs -put -f \
  /mnt/c/data/ratings.csv \
  "/user/$USER/movielens/ratings/"

hdfs dfs -ls -h "/user/$USER/movielens/movies"
hdfs dfs -ls -h "/user/$USER/movielens/ratings"
```

If the CSV files are inside a subdirectory of `C:\data`, change only the two local `/mnt/c/data/...` source paths. Keep the HDFS destinations unchanged.

## 3. Connect Spark to the external Hive catalog

In [ ]:
import os
import socket

from pyspark.sql import SparkSession

master_url = os.environ.get(
    "SPARK_MASTER",
    f"spark://{socket.gethostname()}:7077",
)

spark = (
    SparkSession.builder
    .appName("D30-MovieLens-SQL-Hive")
    .master(master_url)
    .config("hive.metastore.uris", "thrift://localhost:9083")
    .enableHiveSupport()
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")

print("Spark version :", spark.version)
print("Spark master  :", sc.master)
print("Catalog       :", spark.conf.get("spark.sql.catalogImplementation"))
print("Warehouse     :", spark.conf.get("spark.sql.warehouse.dir"))
print("Spark UI      :", sc.uiWebUrl)

## 4. Define explicit schemas

In [ ]:
from pyspark.sql.types import DoubleType, IntegerType, LongType, StringType, StructType

movie_schema = (
    StructType()
    .add("movieId", IntegerType(), nullable=False)
    .add("title", StringType(), nullable=False)
    .add("genres", StringType(), nullable=True)
)

rating_schema = (
    StructType()
    .add("userId", IntegerType(), nullable=False)
    .add("movieId", IntegerType(), nullable=False)
    .add("rating", DoubleType(), nullable=False)
    .add("timestamp", LongType(), nullable=False)
)

## 5. Read MovieLens data from HDFS

In [ ]:
hdfs_user = os.environ["USER"]
movies_path = f"hdfs:///user/{hdfs_user}/movielens/movies"
ratings_path = f"hdfs:///user/{hdfs_user}/movielens/ratings"

movie_df = (
    spark.read
    .option("header", True)
    .schema(movie_schema)
    .csv(movies_path)
)

rating_df = (
    spark.read
    .option("header", True)
    .schema(rating_schema)
    .csv(ratings_path)
)

movie_df.printSchema()
movie_df.show(5, truncate=False)
rating_df.printSchema()
rating_df.show(5)

## 6. Create temporary views

Temporary views exist only in this Spark session. They make DataFrames queryable with SQL, but they are not persistent Hive tables.

In [ ]:
movie_df.createOrReplaceTempView("movies")
rating_df.createOrReplaceTempView("ratings")

spark.sql("SHOW TABLES").show()

## 7. Query the temporary views

`spark.sql` returns a DataFrame, so SQL and DataFrame operations can be combined.

In [ ]:
ratings_sample_df = spark.sql("""
SELECT userId, movieId, rating
FROM ratings
ORDER BY userId, movieId
LIMIT 10
""")

ratings_sample_df.printSchema()
ratings_sample_df.show()

## 8. Aggregate ratings in SQL

The temporary view below contains movies with at least 100 ratings and an average rating of at least 3.5. `HAVING` filters groups after aggregation.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW popular_ratings AS
SELECT
    movieId,
    AVG(rating) AS avg_rating,
    COUNT(*) AS total_ratings
FROM ratings
GROUP BY movieId
HAVING COUNT(*) >= 100
   AND AVG(rating) >= 3.5
""")

spark.sql("""
SELECT *
FROM popular_ratings
ORDER BY total_ratings DESC, avg_rating DESC
LIMIT 20
""").show()

## 9. Join with movie details

In [ ]:
most_popular_movies_df = spark.sql("""
SELECT
    m.movieId,
    m.title,
    m.genres,
    p.avg_rating,
    p.total_ratings
FROM popular_ratings AS p
INNER JOIN movies AS m
    ON p.movieId = m.movieId
ORDER BY p.total_ratings DESC, p.avg_rating DESC
""")

most_popular_movies_df.show(20, truncate=False)

## 10. Save a persistent managed table in Hive

The database and table metadata are stored in the external Hive Metastore. The managed table data is stored under the Hive warehouse on HDFS. Overwrite makes the lesson repeatable.

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS moviedb")

(
    most_popular_movies_df.write
    .mode("overwrite")
    .format("parquet")
    .saveAsTable("moviedb.most_popular_movies")
)

## 11. Query and inspect the Hive table

In [ ]:
spark.sql("SHOW TABLES IN moviedb").show()
spark.sql("SELECT * FROM moviedb.most_popular_movies LIMIT 20").show(truncate=False)
spark.sql("DESCRIBE TABLE EXTENDED moviedb.most_popular_movies").show(100, truncate=False)

## 12. Verify from Beeline (optional)

If HiveServer2 is running, connect from a WSL terminal and query the same persistent table:

```bash
beeline -u 'jdbc:hive2://localhost:10000/default' -n "$USER"
```

```sql
SHOW TABLES IN moviedb;
SELECT * FROM moviedb.most_popular_movies LIMIT 10;
```

The temporary views `movies`, `ratings`, and `popular_ratings` will not appear in Beeline because they belong only to this Spark session.

## 13. Stop Spark

The persistent Hive table remains available after this session stops.

In [ ]:
spark.stop()
print("Spark session stopped.")